# 因子研究模板

这个notebook是因子研究的模板，你可以复制这个文件夹来创建新的因子。

## 使用步骤：
1. 复制 `factor_template` 文件夹，重命名为你的因子名称
2. 在下面的代码单元格中实现你的因子逻辑
3. 运行分析查看因子表现
4. 调整参数优化因子

In [ ]:
import sys
sys.path.insert(0, '/home/day_strategy/factors/utils')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from factor_utils import (
    load_symbol, load_all_symbols, normalize_factor,
    analyze_factor_distribution, analyze_factor_ic,
    quick_backtest_single, analyze_backtest_results,
    quick_factor_analysis
)

%matplotlib inline

## 1. 定义因子

在下面实现你的因子计算逻辑：

In [ ]:
def calculate_my_factor(df, lookback=20, normalize_method='zscore'):
    """
    计算自定义因子
    
    Args:
        df: 包含日线数据的DataFrame
        lookback: 回看周期
        normalize_method: 标准化方法 ('zscore', 'rank', 'mad', 'none')
    
    Returns:
        因子值的Series
    """
    # TODO: 在这里实现你的因子逻辑
    # 示例：简单动量因子
    raw_factor = df['close'].pct_change(lookback)
    
    # 标准化（使分布稳定）
    factor = normalize_factor(raw_factor, method=normalize_method)
    
    return factor

# 因子名称（用于输出）
FACTOR_NAME = "MyFactor"

## 2. 单品种分析

先在一个品种上测试因子效果：

In [ ]:
# 选择测试品种
TEST_SYMBOL = 'AG'  # 可以改为其他品种

# 快速分析（包含分布、IC、回测）
factor, trades = quick_factor_analysis(
    symbol=TEST_SYMBOL,
    factor_func=calculate_my_factor,
    lookback=20,
    normalize_method='zscore'
)

## 3. 参数敏感性分析

测试不同参数下的因子表现：

In [ ]:
# 加载数据
df = load_symbol(TEST_SYMBOL)

# 测试不同lookback
lookbacks = [5, 10, 20, 40, 60]
results = []

for lb in lookbacks:
    factor = calculate_my_factor(df, lookback=lb)
    trades = quick_backtest_single(factor, df)
    
    if len(trades) > 0:
        results.append({
            'lookback': lb,
            'total_trades': len(trades),
            'win_rate': (trades['pnl'] > 0).mean(),
            'avg_return': trades['pnl'].mean(),
            'sharpe': trades['pnl'].mean() / trades['pnl'].std() * np.sqrt(252) if trades['pnl'].std() > 0 else 0,
            'total_pnl': (1 + trades['pnl']).prod() - 1
        })

param_df = pd.DataFrame(results)
print(param_df)

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(param_df['lookback'], param_df['sharpe'], 'o-')
axes[0, 0].set_title('Sharpe vs Lookback')
axes[0, 0].set_xlabel('Lookback')
axes[0, 0].set_ylabel('Sharpe')
axes[0, 0].grid(True)

axes[0, 1].plot(param_df['lookback'], param_df['win_rate'], 'o-')
axes[0, 1].set_title('Win Rate vs Lookback')
axes[0, 1].set_xlabel('Lookback')
axes[0, 1].set_ylabel('Win Rate')
axes[0, 1].grid(True)

axes[1, 0].plot(param_df['lookback'], param_df['avg_return'], 'o-')
axes[1, 0].set_title('Avg Return vs Lookback')
axes[1, 0].set_xlabel('Lookback')
axes[1, 0].set_ylabel('Avg Return')
axes[1, 0].grid(True)

axes[1, 1].plot(param_df['lookback'], param_df['total_pnl'], 'o-')
axes[1, 1].set_title('Total PnL vs Lookback')
axes[1, 1].set_xlabel('Lookback')
axes[1, 1].set_ylabel('Total PnL')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## 4. 多品种回测

在所有品种上测试因子：

In [ ]:
# 加载所有品种
all_symbols = load_all_symbols()
print(f"加载了 {len(all_symbols)} 个品种")

# 回测配置
ENTRY_THRESHOLD = 1.0
EXIT_THRESHOLD = 0.0
MAX_HOLDING_DAYS = 10
LOOKBACK = 20

# 运行回测
all_results = []

for symbol, df in all_symbols.items():
    try:
        factor = calculate_my_factor(df, lookback=LOOKBACK)
        trades = quick_backtest_single(
            factor, df,
            entry_threshold=ENTRY_THRESHOLD,
            exit_threshold=EXIT_THRESHOLD,
            max_holding_days=MAX_HOLDING_DAYS
        )
        
        if len(trades) > 0:
            all_results.append({
                'symbol': symbol,
                'total_trades': len(trades),
                'win_rate': (trades['pnl'] > 0).mean(),
                'avg_return': trades['pnl'].mean(),
                'sharpe': trades['pnl'].mean() / trades['pnl'].std() * np.sqrt(252) if trades['pnl'].std() > 0 else 0,
                'total_pnl': (1 + trades['pnl']).prod() - 1,
                'max_dd': ((1 + trades['pnl']).cumprod() - (1 + trades['pnl']).cumprod().cummax()).min()
            })
    except Exception as e:
        print(f"Error processing {symbol}: {e}")

results_df = pd.DataFrame(all_results)
print(f"\n成功回测 {len(results_df)} 个品种")

# 显示汇总
print("\n绩效汇总:")
print(results_df[['symbol', 'total_trades', 'win_rate', 'avg_return', 'sharpe', 'total_pnl']].sort_values('sharpe', ascending=False))

## 5. 多品种绩效可视化

查看因子在所有品种上的表现分布：

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(f'{FACTOR_NAME} Multi-Symbol Results', fontsize=14)

# 夏普比率分布
axes[0, 0].hist(results_df['sharpe'], bins=15, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(results_df['sharpe'].mean(), color='r', linestyle='--', 
                   label=f"Mean: {results_df['sharpe'].mean():.2f}")
axes[0, 0].set_title('Sharpe Ratio Distribution')
axes[0, 0].set_xlabel('Sharpe')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

# 胜率分布
axes[0, 1].hist(results_df['win_rate'], bins=15, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(results_df['win_rate'].mean(), color='r', linestyle='--',
                   label=f"Mean: {results_df['win_rate'].mean():.2%}")
axes[0, 1].set_title('Win Rate Distribution')
axes[0, 1].set_xlabel('Win Rate')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

# 平均收益分布
axes[0, 2].hist(results_df['avg_return'], bins=15, edgecolor='black', alpha=0.7)
axes[0, 2].axvline(results_df['avg_return'].mean(), color='r', linestyle='--',
                   label=f"Mean: {results_df['avg_return'].mean():.4f}")
axes[0, 2].set_title('Avg Return Distribution')
axes[0, 2].set_xlabel('Avg Return')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].legend()

# 总收益分布
axes[1, 0].hist(results_df['total_pnl'], bins=15, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(results_df['total_pnl'].mean(), color='r', linestyle='--',
                   label=f"Mean: {results_df['total_pnl'].mean():.2%}")
axes[1, 0].set_title('Total PnL Distribution')
axes[1, 0].set_xlabel('Total PnL')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

# 交易次数分布
axes[1, 1].hist(results_df['total_trades'], bins=15, edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Trade Count Distribution')
axes[1, 1].set_xlabel('Trade Count')
axes[1, 1].set_ylabel('Frequency')

# 夏普 vs 胜率
axes[1, 2].scatter(results_df['win_rate'], results_df['sharpe'], alpha=0.6)
axes[1, 2].set_xlabel('Win Rate')
axes[1, 2].set_ylabel('Sharpe Ratio')
axes[1, 2].set_title('Sharpe vs Win Rate')

plt.tight_layout()
plt.show()

# 打印汇总统计
print(f"\n{'='*60}")
print(f"因子: {FACTOR_NAME}")
print(f"{'='*60}")
print(f"品种数量: {len(results_df)}")
print(f"平均交易次数: {results_df['total_trades'].mean():.1f}")
print(f"平均胜率: {results_df['win_rate'].mean():.2%}")
print(f"平均收益: {results_df['avg_return'].mean():.4f}")
print(f"平均夏普: {results_df['sharpe'].mean():.2f}")
print(f"平均总收益: {results_df['total_pnl'].mean():.2%}")
print(f"{'='*60}")

## 6. 保存结果

保存回测结果供后续分析：

In [ ]:
from pathlib import Path

# 创建输出目录
output_dir = Path(f'/home/day_strategy/factors/output/{FACTOR_NAME}')
output_dir.mkdir(parents=True, exist_ok=True)

# 保存结果
results_df.to_csv(output_dir / 'backtest_summary.csv', index=False)

# 保存图表
fig.savefig(output_dir / 'analysis.png', dpi=150, bbox_inches='tight')

print(f"结果已保存至: {output_dir}")